# Route definition 

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]


# IVA calculations



In [2]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_gastos_2018= pd.read_csv(
    RAW_DIR / "gastoshogar_2018.csv",
    dtype={
        "folioviv": str,
        "foliohog": str,
        "clave": str
    },
    low_memory=False
)

# ==========================================================
# Limpiar variables
# ==========================================================

df_gastos_2018["gasto_tri"] = pd.to_numeric(
    df_gastos_2018["gasto_tri"],
    errors="coerce"
).fillna(0)

# Sólo gastos monetarios
df_gastos_2018 = df_gastos_2018[df_gastos_2018["tipo_gasto"] == "G1"]

# Sólo compras formales
informal_locs = [1, 2, 3, 17]

df_gastos_2018["es_formal"] = (
    ~df_gastos_2018["lugar_comp"].isin(informal_locs)
).astype(int)

# ==========================================================
# Identificar productos con IVA
# ==========================================================

CLAVES_IVA_CERO_2018 = [
    # Cereales y derivados (básicos)
    'A001', 'A002', 'A003', 'A004', 'A007', 'A008', 'A019',
    # Carnes (res, puerco, pollo, pescado)
    'A025', 'A026', 'A027', 'A028', 'A029', 'A030', 'A031', 'A032', 'A033', 'A034', 'A035', 'A036', 'A037',
    'A038', 'A039', 'A040', 'A041', 'A042', 'A043', 'A044', 'A045', 'A046',
    'A057', 'A058', 'A059', 'A060', 'A061',
    'A066', 'A067', 'A068', 'A069', 'A070',
    # Leche y huevo
    'A075', 'A076', 'A077', 'A078', 'A079', 'A080', 'A081', 'A093',
    # Aceites
    'A095', 'A096',
    # Verduras, frutas y legumbres
    'A101', 'A102', 'A103', 'A104', 'A107', 'A108', 'A109', 'A110', 'A111', 'A112',
    'A113', 'A114', 'A115', 'A116', 'A117', 'A118', 'A119', 'A120', 'A121', 'A122',
    'A123', 'A124', 'A125', 'A126', 'A127', 'A128', 'A129', 'A130', 'A131',
    'A137', 'A138', 'A139', 'A140', 'A141',
    'A147', 'A148', 'A149', 'A150', 'A151', 'A152', 'A153', 'A154', 'A155', 'A156',
    'A157', 'A158', 'A159', 'A160', 'A161', 'A162', 'A163', 'A164', 'A165', 'A166',
    'A167', 'A168', 'A169', 'A170',
    # Azúcar y sal
    'A173', 'A191',
    # Agua natural
    'A215',
    # Transporte público
    'B001', 'B002', 'B003', 'B004', 'B005', 'B006', 'B007',
    # Educación (colegiaturas y servicios)
    'E001', 'E002', 'E003', 'E004', 'E005', 'E006', 'E007', 'E008', 'E009', 'E010', 'E011',
    # Libros, periódicos, revistas
    'E022', 'E023', 'E024',
    # Renta de vivienda
    'G101', 'G102', 'G103', 'G104', 'G105', 'G106',
    # Servicios médicos y medicamentos recetados
    'J007', 'J008', 'J016', 'J017', 'J018', 'J039',
    'J020', 'J021', 'J022', 'J023', 'J024', 'J025', 'J026', 'J027', 'J028', 'J029',
    'J030', 'J031', 'J032', 'J033', 'J034', 'J035',
    'J044', 'J045', 'J046', 'J047', 'J048', 'J049', 'J050', 'J051', 'J052', 'J053',
    'J054', 'J055', 'J056', 'J057', 'J058', 'J059',
]

df_gastos_2018["lleva_iva"] = (
    ~df_gastos_2018["clave"].isin(CLAVES_IVA_CERO_2018)
).astype(int)

# ==========================================================
# Separar gasto gravado y no gravado
# ==========================================================

df_gastos_2018["gasto_iva"] = np.where(
    (df_gastos_2018["lleva_iva"] == 1) &
    (df_gastos_2018["es_formal"] == 1),
    df_gastos_2018["gasto_tri"],
    0
)

df_gastos_2018["gasto_no_iva"] = np.where(
    df_gastos_2018["lleva_iva"] == 0,
    df_gastos_2018["gasto_tri"],
    0
)

# ==========================================================
# IVA pagado (16% incluido en el precio)
# ==========================================================

FACTOR_IVA = 16 / 116

df_gastos_2018["iva_pagado"] = df_gastos_2018["gasto_iva"] * FACTOR_IVA

# ==========================================================
# Resumen por hogar
# ==========================================================

iva_hogar_2018 = (
    df_gastos_2018
    .groupby(["folioviv", "foliohog"], as_index=False)
    .agg(
        gasto_con_iva=("gasto_iva", "sum"),
        gasto_sin_iva=("gasto_no_iva", "sum"),
        iva_pagado=("iva_pagado", "sum"),
        gasto_total=("gasto_tri", "sum")
    )
)

iva_hogar.head()

NameError: name 'iva_hogar' is not defined

In [6]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_gastos_2024= pd.read_csv(
    RAW_DIR / "gastoshogar_2024.csv",
    dtype={
        "folioviv": str,
        "foliohog": str,
        "clave": str
    },
    low_memory=False
)

# ==========================================================
# Limpiar variables
# ==========================================================

df_gastos_2024["gasto_tri"] = pd.to_numeric(
    df_gastos_2024["gasto_tri"],
    errors="coerce"
).fillna(0)

# Sólo gastos monetarios
df_gastos_2024 = df_gastos_2024[df_gastos_2024["tipo_gasto"] == "G1"]

# Sólo compras formales
informal_locs = [1, 2, 3, 17]

df_gastos_2024["es_formal"] = (
    ~df_gastos_2024["lugar_comp"].isin(informal_locs)
).astype(int)

# ==========================================================
# Identificar productos con IVA
# ==========================================================

CLAVES_IVA_CERO_2024 = [
    # 01: Alimentos y bebidas no alcohólicas (tasa 0% para básicos)
    '011111', '011112', '011113',  # Arroz, maíz, otros cereales
    '011121', '011122',           # Harinas de maíz y trigo
    '011131', '011132',           # Tortillas
    '011133',                     # Pan (general)
    '011191', '011192', '011193', # Tostadas, masa
    '011210',                     # Animales vivos
    '011221', '011222', '011223', '011224', '011225', '011226', '011227',  # Carnes de res
    '011228', '011229', '01122A', '01122B', '01122C',                     # Carnes de puerco
    '01122D',                     # Cordero y cabra
    '01122F', '01122G',           # Pollo y otras aves
    '01122H', '01122I',           # Otras carnes e insectos
    '011241', '011242', '011243', '011244', # Vísceras
    '011245', '011246', '011247',           # Otras partes de pollo y res
    '011310',                     # Pescado fresco
    '011331',                     # Filetes empanizados
    '011341', '011342', '011343', # Camarón y mariscos
    '011370',                     # Huevas y despojos de mariscos
    '011411', '011412', '011420', # Leche fresca y descremada
    '011432',                     # Leche en polvo
    '011433',                     # Crema
    '011441', '011442',           # Leche de soya y otras
    '011451', '011452', '011454', '011455', '011456',  # Quesos (fresco, amarillo, Oaxaca, añejo, otros)
    '011460',                     # Yogurt
    '011471',                     # Bebidas de leche
    '011481', '011482',           # Huevo de gallina y otros
    '011490',                     # Otros lácteos
    '011511', '011512',           # Aceite de oliva y otros
    '011520',                     # Mantequilla
    '011530',                     # Margarina
    '011591',                     # Manteca de puerco
    '011592',                     # Chicharrón
    '011593',                     # Otras grasas animales
    '011611', '011612', '011613', '011614', '011615', '011616', '011617',  # Frutas tropicales
    '011621', '011622', '011623', '011624',  # Cítricos
    '011631', '011632', '011633', '011634', '011635',  # Frutas de hueso
    '011641', '011642',           # Fresas y bayas
    '011651', '011652', '011653', '011654',  # Melón, sandía, uva, otras
    '011660',                     # Fruta congelada
    '011670',                     # Frutos secos
    '011680',                     # Nueces
    '011711', '011712', '011713',  # Hortalizas de hoja
    '011721', '011722', '011723', '011724', '011725', '011726',  # Calabaza, chayote, chiles
    '011727',                     # Jitomate
    '011728',                     # Pepino
    '011729',                     # Tomate verde
    '01172A',                     # Otras hortalizas frutales
    '011731', '011732', '011733',  # Chícharo, ejote, otras legumbres
    '011741', '011742', '011743', '011744', '011745', '011746', '011747', '011748', '011749',  # Otras verduras
    '011751',                     # Papa
    '011752',                     # Betabel
    '011753',                     # Jícama
    '011754',                     # Plátano macho
    '011755',                     # Otros tubérculos
    '011761', '011762', '011763', '011764',  # Frijol, lentejas, garbanzo, otras legumbres
    '011770',                     # Hortalizas secas
    '011781', '011782',           # Verduras y legumbres congeladas
    '011791',                     # Harina de papa
    '011793',                     # Chiles secos
    '011796',                     # Papas fritas (tasa 0% si son básicas)
    '011797',                     # Frijol procesado
    '011810',                     # Azúcar
    '011820',                     # Sustitutos de azúcar
    '011831',                     # Miel de abeja
    '011832',                     # Otras mieles
    '011833',                     # Mermeladas
    '011841', '011842', '011843',  # Cremas de cacahuate y otras
    '011861',                     # Helados
    '011862',                     # Hielo
    '011937',                     # Sal
    '012501',                     # Agua natural embotellada
    # 02: Bebidas alcohólicas y tabaco (generalmente 16%, pero algunos pueden tener 0%)
    # 03: Vestido y calzado (generalmente 16%)
    # 04: Vivienda (tasa 0% para rentas)
    '046261', '046262', '046263', '046264', '046265', '046266',  # Renta y estimaciones de alquiler
    # 05: Mobiliario (generalmente 16%)
    # 06: Salud (tasa 0% para medicinas y servicios médicos)
    '061111', '061112', '061113', '061114', '061115', '061116', '061117',
    '061118', '061119', '06111A', '06111B', '06111C', '06111D', '06111E',
    '06111F', '06111G', '06111H', '06111I', '06111J', '06111K', '06111L',  # Medicamentos
    '061120',                     # Productos homeopáticos y naturistas
    '061211', '061212',           # Pruebas de embarazo, termómetros, glucómetros
    '061221',                     # Preservativos
    '061222',                     # Parches de nicotina
    '061230',                     # Productos de tratamiento (jeringas, etc.)
    '061311',                     # Lentes
    '062110',                     # Vacunación
    '062191', '062192', '062193', '062194', '062195', '062196', '062197', '062198',  # Consultas médicas
    '062210',                     # Consultas dentales
    '062291', '062292',           # Ortodoncia y prótesis dentales
    '062311', '062312', '062313',  # Servicios curativos y rehabilitación
    '062321', '062322', '062323',  # Servicios de enfermería y asistencia
    '063101', '063102', '063103', '063104', '063105', '063106',  # Hospitalización
    '063200',                     # Hospitalización en salud mental
    '064101', '064102', '064103',  # Análisis clínicos y estudios
    '064201', '064202',           # Servicios de ambulancia
    # 07: Transporte (tasa 0% para pasajes)
    '073111', '073112',           # Tren
    '073121', '073122', '073123',  # Metro, tren ligero, trolebús
    '073211', '073212', '073213', '073214', '073215',  # Autobuses y colectivos
    '073221',                     # Taxi (puede tener 16% en algunos casos)
    '073222',                     # Servicios de transporte privado (Uber, etc.)
    '073230',                     # Transporte escolar
    '073290',                     # Otro transporte por carretera
    '073310', '073320',           # Vuelos nacionales e internacionales (pueden tener 16%)
    '073401', '073402',           # Transporte marítimo
    '073500',                     # Transporte combinado
    '073600',                     # Funicular, teleférico
    # 08: Comunicaciones (generalmente 16%)
    # 09: Recreación y cultura (tasa 0% para libros, periódicos, revistas)
    '097111',                     # Libros de texto
    '097191',                     # Libros de ficción
    '097210',                     # Periódicos
    '097220',                     # Revistas
    # 10: Educación (tasa 0% para servicios educativos)
    '101011', '101012', '101013', '101014', '101015',  # Preescolar
    '101021', '101022', '101023', '101024', '101025',  # Primaria
    '102001', '102002', '102003', '102004', '102005',  # Secundaria
    '103001', '103002', '103003', '103004', '103005',  # Bachillerato
    '104011', '104012', '104013', '104014',           # Carrera técnica
    '104015', '104016', '104017',                     # Educación superior
    '104018', '104019', '10401A',                     # Posgrado
    '105010',                     # Tutorías
    '105091', '105092', '105093',  # Educación para el trabajo
    # 11: Restaurantes y hoteles (generalmente 16%, pero algunos servicios pueden tener 0%)
    # 12: Seguros y servicios financieros (generalmente 16%)
    # 13: Cuidado personal (generalmente 16%)
    # 17: Gastos diversos (algunos pueden tener 0%)
    '173133', '173134',           # Materiales y servicios para construcción (pueden tener 16%)
    '173411',                     # Contribuciones para obras públicas (pueden tener 0%)
    '173416',                     # Ayuda a parientes (puede tener 0%)
    '173417',                     # Donativos (pueden tener 0%)
    # T: Regalos (la tasa depende del producto)
]

df_gastos_2024["lleva_iva"] = (
    ~df_gastos_2024["clave"].isin(CLAVES_IVA_CERO_2024)
).astype(int)

# ==========================================================
# Separar gasto gravado y no gravado
# ==========================================================

df_gastos_2024["gasto_iva"] = np.where(
    (df_gastos_2024["lleva_iva"] == 1) &
    (df_gastos_2024["es_formal"] == 1),
    df_gastos_2024["gasto_tri"],
    0
)

df_gastos_2024["gasto_no_iva"] = np.where(
    df_gastos_2024["lleva_iva"] == 0,
    df_gastos_2024["gasto_tri"],
    0
)

# ==========================================================
# IVA pagado (16% incluido en el precio)
# ==========================================================

FACTOR_IVA = 16 / 116

df_gastos_2024["iva_pagado"] = df_gastos_2024["gasto_iva"] * FACTOR_IVA

# ==========================================================
# Resumen por hogar
# ==========================================================

iva_hogar_2024 = (
    df_gastos_2024
    .groupby(["folioviv", "foliohog"], as_index=False)
    .agg(
        gasto_con_iva=("gasto_iva", "sum"),
        gasto_sin_iva=("gasto_no_iva", "sum"),
        iva_pagado=("iva_pagado", "sum"),
        gasto_total=("gasto_tri", "sum")
    )
)

iva_hogar_2024.head()

,folioviv,foliohog,gasto_con_iva,gasto_sin_iva,iva_pagado,gasto_total
0,0100001901,1,30975.11,8344.67,4272.428966,39988.34
1,0100001902,1,26041.44,10427.04,3591.922759,36468.48
2,0100001904,1,16618.46,8644.97,2292.201379,28601.26
3,0100001905,1,30382.94,12812.31,4190.750345,43683.82
4,0100002501,1,82446.22,27668.50,11371.892414,124056.39


# Gastos en gasolina y electricidad 

In [ ]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_gastos_2018= pd.read_csv(
    RAW_DIR / "gastoshogar_2018.csv",
    dtype={
        "folioviv": str,
        "foliohog": str,
        "clave": str
    },
    low_memory=False
)

df_gastos_2024["gasto_tri"] = pd.to_numeric(
    df_gastos_2024["gasto_tri"],
    errors="coerce"
).fillna(0)

# Sólo gastos monetarios
df_gastos_2024 = df_gastos_2024[df_gastos_2024["tipo_gasto"] == "G1"]


In [3]:
import pandas as pd
import numpy as np

# ==========================================================
# Cargar base de gastos
# ==========================================================

df_gastos_gas_2024= pd.read_csv(
    RAW_DIR / "gastoshogar_2024.csv",
    dtype={
        "folioviv": str,
        "foliohog": str,
        "clave": str
    },
    low_memory=False
)

df_gastos_gas_2024["gasto_tri"] = pd.to_numeric(
    df_gastos_gas_2024["gasto_tri"],
    errors="coerce"
).fillna(0)

# Sólo gastos monetarios
df_gastos_gas_2024 = df_gastos_gas_2024[df_gastos_gas_2024["tipo_gasto"] == "G1"]


df_gastos_gas_2024.head()

,folioviv,foliohog,clave,tipo_gasto,mes_dia,forma_pag1,forma_pag2,forma_pag3,lugar_comp,orga_inst,...,num_pagos,ultim_pago,gasto_tri,gasto_nm,gas_nm_tri,imujer_tri,entidad,est_dis,upm,factor
0,0100001901,1,011131,G1,1031,1,0,0,4,0,...,,,167.14,,,,1,1,1,207
1,0100001901,1,011150,G1,1029,1,0,0,4,0,...,,,642.85,,,,1,1,1,207
2,0100001901,1,011231,G1,1031,1,0,0,4,0,...,,,192.85,,,,1,1,1,207
3,0100001901,1,011411,G1,1029,1,0,0,4,0,...,,,488.57,,,,1,1,1,207
4,0100001901,1,011460,G1,1029,1,0,0,4,0,...,,,1465.71,,,,1,1,1,207


# Educación y salud 